# Lenguajes y Autómatas II (SCD-1016)
## Sesión 36 · Traducción de Comparaciones y Saltos a Ensamblador

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se convierte un if o un while en un salto que la CPU sepa ejecutar?
>
> **Objetivo de la sesión:** Elegir la instrucción de salto correcta para cada comparación, con y sin signo, y comprobar cuándo la condición se cumple.

**Entregable:** tu implementación de salto_para() y evaluar_salto() en artesano_sesion36_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion36.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes un operador relacional y una bandera de signo, o dos valores comparados con ese operador; devuelves el salto AT&T correcto o el resultado de la comparación.

Al final de la sesión tendrás: **Una función que traduce operadores a saltos condicionales y otra que evalúa la comparación con la misma semántica que la CPU.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 36](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion36.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion36.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion36.h"

const char *salto_para(const char *operador, int con_signo);
int evaluar_salto(const char *operador, long a, long b);
```

- Los operadores válidos son <, <=, >, >=, == y !=.
- Con signo se usan jl, jle, jg y jge; sin signo, jb, jbe, ja y jae; igualdad y diferencia no cambian.
- Un operador desconocido responde nop y una comparación desconocida evalúa a falso.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion36_c.c` junto con `artesano_sesion36_c.c`
> (o `ia_sesion36_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `salto_para("<", 1)` | `jl` |
| 2 | `salto_para("<", 0)` | `jb` |
| 3 | `salto_para("==", 0)` | `je` |
| 4 | `evaluar_salto("<", 3, 5)` | 1 |
| 5 | `evaluar_salto(">=", 5, 5)` | 1 |
| 6 | `evaluar_salto("!=", 7, 7)` | 0 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 36 - Instrucciones cmp, jmp y saltos condicionales ===
Caso                               esperado:              obtenido:              resultado
[ 1] salto_para("<", 1)                 esperado: jl           obtenido: jl           OK
[ 2] salto_para("<", 0)                 esperado: jb           obtenido: jb           OK
[ 3] salto_para("==", 0)                esperado: je           obtenido: je           OK
[ 4] evaluar_salto("<", 3, 5)           esperado: 1            obtenido: 1            OK
[ 5] evaluar_salto(">=", 5, 5)          esperado: 1            obtenido: 1            OK
[ 6] evaluar_salto("!=", 7, 7)          esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion36.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] salto_para distingue correctamente los operadores con signo de los que no lo tienen.
- [ ] evaluar_salto devuelve 1 solo cuando la comparación es verdadera, igual que las banderas de cmp.
- [ ] Los seis casos del harness salen OK y el programa termina con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion36_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion36.h
/* api_sesion36.h - contrato de la Sesion 36
 * Instrucciones cmp, jmp y saltos condicionales
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION36_H
#define API_SESION36_H

const char *salto_para(const char *operador, int con_signo);
int evaluar_salto(const char *operador, long a, long b);

#endif /* API_SESION36_H */


In [ ]:
%%writefile artesano_sesion36_c.c
/* artesano_sesion36_c.c - Fase 1: El Artesano (sin IA)
 * Instrucciones cmp, jmp y saltos condicionales.
 *
 * Implementa lo que declara api_sesion36.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion36.h"
#include <stdio.h>
#include <string.h>

const char *salto_para(const char *operador, int con_signo) {
    (void)operador;
    (void)con_signo; /* TODO: mapear el operador al salto condicional */
    return "nop";
}

int evaluar_salto(const char *operador, long a, long b) {
    (void)operador;
    (void)a;
    (void)b; /* TODO: comparar como lo haria cmp */
    return 0;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion36_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion36_c.c
/* ia_sesion36_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion36.h. No puede traer main().
 */
#include "api_sesion36.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion36_c.c
/* harness_sesion36_c.c - auditoria automatica de la Sesion 36
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion36_c.c artesano_sesion36_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion36.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 36 - Instrucciones cmp, jmp y saltos condicionales ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { const char *r = salto_para("<", 1); verificar_str("salto_para(\"<\", 1)", "jl", r); }
    { const char *r = salto_para("<", 0); verificar_str("salto_para(\"<\", 0)", "jb", r); }
    { const char *r = salto_para("==", 0); verificar_str("salto_para(\"==\", 0)", "je", r); }
    { int r = evaluar_salto("<", 3, 5); verificar_int("evaluar_salto(\"<\", 3, 5)", (long)(1), (long)(r)); }
    { int r = evaluar_salto(">=", 5, 5); verificar_int("evaluar_salto(\">=\", 5, 5)", (long)(1), (long)(r)); }
    { int r = evaluar_salto("!=", 7, 7); verificar_int("evaluar_salto(\"!=\", 7, 7)", (long)(0), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion36.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion36_c.c artesano_sesion36_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion36_c.c ia_sesion36_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion36.md` a `auditorias/auditoria_sesion36.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
